# Setup and the course LLM client

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import sys
print(sys.executable)

/Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2/.venv/bin/python


In [2]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


Run this notebook first. It checks your environment, seeds the replay cache from the recorded responses in `data/llm_fixtures/`, and shows the client from the orientation slides (parts 1 to 4) at work. Replay mode needs no API key.

In [3]:
import importlib
for pkg in ["yaml", "numpy", "pandas", "sklearn", "openpyxl", "bs4", "pydantic", "duckdb", "chromadb", "rank_bm25", "langchain_text_splitters", "fastapi", "matplotlib"]:
    try:
        importlib.import_module(pkg); print(f"ok       {pkg}")
    except ImportError:
        print(f"MISSING  {pkg}  ->  pip install -r requirements.txt")

ok       yaml
ok       numpy
ok       pandas


ok       sklearn
ok       openpyxl
ok       bs4
ok       pydantic
ok       duckdb


ok       chromadb
ok       rank_bm25
ok       langchain_text_splitters
ok       fastapi


ok       matplotlib


In [4]:
import subprocess
subprocess.run([sys.executable, "tools/seed_replay_cache.py"], check=True)

seeded 12 model responses into data/llm_cache and 34 embeddings into data/embed_cache


CompletedProcess(args=['/Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2/.venv/bin/python', 'tools/seed_replay_cache.py'], returncode=0)

`src/llm/client.py` is the slides' code (parts 1, 3, 2, 4 in dependency order) plus a small adapter for live calls. Every model and embedding call in Case Study 2 goes through it.

In [5]:
import inspect
print(inspect.getsource(llm.structured))

def structured(response_model, messages, model,
               stage="build", **kw):
    schema = response_model.model_json_schema()
    key = cache_key(PROVIDER, model, messages,
                    schema=schema, **kw)
    f = CACHE / f"{key}.json"
    if f.exists():                          # hit
        return response_model.model_validate_json(
            f.read_text())
    if MODE == "replay":
        raise RuntimeError(f"cache miss {key}")
    # budget_check and ledger are defined in part 3
    budget_check(model, len(json.dumps(messages)) // 4)
    out, raw = INSTR.create_with_completion(
        response_model=response_model, model=model,
        messages=messages, **kw)
    if MODE == "record":          # cache before ledger,
        f.write_text(out.model_dump_json())  # so no re-pay
    u = raw.usage                 # names vary by provider
    ledger(model, u.prompt_tokens,
           u.completion_tokens, stage)
    return out



In [6]:
# A replay hit: the recorded claim extraction for one CIM block
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
out = llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks["p0002_b001"]["text"]), llm.P["llm"]["models"]["large"])
for c in out.claims: print(c.metric, c.value, c.unit, c.period, c.qualifier)

revenue_growth 0.32 pct FY2025 reported
revenue 41.2 USD FY2025 reported


In [7]:
# A replay miss is an error that names the missing key: replay never calls a provider
try:
    llm.structured(schemas.MgmtClaims, prompts.claims_messages("A sentence nobody recorded."), llm.P["llm"]["models"]["large"])
except RuntimeError as e:
    print("as expected:", e)

as expected: cache miss 84864a8fa130c8583aa5049ca783e8d6554a68b366909fc8366eda7361205371


In [8]:
# The cost ledger and the budget cap (part 3): what a live call of 1,000 tokens in and 300 out would cost
print("USD:", llm.usd(llm.P["llm"]["models"]["large"], 1000, 300), "| cap:", llm.CAP, "| spent so far:", llm.spent())

USD: 0.00088 | cap: 5.0 | spent so far: 0.00021


**Exercise 0 — record once, then prove offline replay.**

The custom sentence below already has a recorded response in this repository. The default replay run must use that cache without loading `.env`, requiring credentials, or adding cost.

To record a different sentence intentionally, start Jupyter with `LANTERN_LLM_MODE=record` and restart the kernel. Only record mode loads `.env` for credentials. A new request should create one cache file and one ledger row; repeating it must add neither. Existing cache hits must add zero ledger rows.

The final cell starts a fresh Python process with API keys removed and network connections blocked, compares the response, and verifies that the ledger is unchanged.

In [9]:
model = "openai/gpt-4.1-mini"
exercise_mode = llm.MODE
assert exercise_mode in {"record", "replay"}, "Use record or replay for this exercise."

if exercise_mode == "record":
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
    assert os.getenv("OPENAI_API_KEY"), "Record mode requires OPENAI_API_KEY."
    assert os.getenv("OPENAI_BASE_URL") == "https://openrouter.ai/api/v1"
    # Retain the pricing used for the original classroom recording.
    llm.PRICES[model] = [0.40, 1.60]
    llm.INSTR._client = None
    print("Record mode selected; credentials are not displayed.")
else:
    print("Replay mode: .env is not loaded and no API key is required.")

Replay mode: .env is not loaded and no API key is required.


In [10]:
messages = prompts.claims_messages(
    "Practice company reported FY2025 revenue of USD 12 million."
)
key = llm.cache_key(
    llm.PROVIDER, model, messages,
    schema=schemas.MgmtClaims.model_json_schema(),
)
cache_file = llm.CACHE / f"{key}.json"

def ledger_rows():
    path = Path(llm.LEDGER)
    return len(path.read_text().splitlines()) if path.exists() else 0

was_cached = cache_file.exists()
before = ledger_rows()
try:
    recorded = llm.structured(
        schemas.MgmtClaims, messages, model, stage="exercise_0"
    )
    after_first = ledger_rows()
    expected_new_rows = int(exercise_mode == "record" and not was_cached)
    assert cache_file.exists()
    assert after_first - before == expected_new_rows

    repeated = llm.structured(
        schemas.MgmtClaims, messages, model, stage="exercise_0"
    )
    assert repeated.model_dump() == recorded.model_dump()
    assert ledger_rows() == after_first
    print(recorded.model_dump_json(indent=2))
    print("Cache file exists:", cache_file.exists())
    print("Already cached before this run:", was_cached)
    print("New ledger rows:", after_first - before)
    print("Second call: identical result; no additional ledger row.")
finally:
    llm.MODE = "replay"

{
  "claims": [
    {
      "metric": "revenue",
      "value": 12.0,
      "unit": "USD",
      "scale": 1000000,
      "period": "FY2025",
      "qualifier": "reported",
      "quote": "Practice company reported FY2025 revenue of USD 12 million."
    }
  ]
}
Cache file exists: True
Already cached before this run: True
New ledger rows: 0
Second call: identical result; no additional ledger row.


In [11]:
# A separate process provides a fresh client, without inherited notebook state.
import subprocess

child_env = os.environ.copy()
for name in list(child_env):
    if "API_KEY" in name or name == "OPENAI_BASE_URL":
        child_env.pop(name)
child_env["LANTERN_LLM_MODE"] = "replay"

proof = r"""
import json, os, socket
from unittest.mock import patch
from src.llm import client as llm
from src import schemas, prompts
assert llm.MODE == "replay"
assert not any("API_KEY" in name for name in os.environ)
messages = prompts.claims_messages(
    "Practice company reported FY2025 revenue of USD 12 million."
)
with patch.object(socket.socket, "connect", side_effect=AssertionError("Network forbidden")), \
     patch.object(socket, "create_connection", side_effect=AssertionError("Network forbidden")), \
     patch.object(llm.INSTR, "create_with_completion", side_effect=AssertionError("Provider call forbidden")):
    result = llm.structured(
        schemas.MgmtClaims, messages, "openai/gpt-4.1-mini", stage="exercise_0"
    )
print(result.model_dump_json())
"""
ledger_path = Path(llm.LEDGER)
ledger_before = ledger_path.read_bytes() if ledger_path.exists() else None
result = subprocess.run(
    [sys.executable, "-c", proof], cwd=ROOT, env=child_env,
    capture_output=True, text=True, check=True,
)
assert json.loads(result.stdout) == recorded.model_dump()
ledger_after = ledger_path.read_bytes() if ledger_path.exists() else None
assert ledger_after == ledger_before
print("Fresh-process replay passed: API keys unset, network/provider calls blocked.")
print("Recorded response matches; cost ledger unchanged.")

Fresh-process replay passed: API keys unset, network/provider calls blocked.
Recorded response matches; cost ledger unchanged.
